In [ ]:
import os
import json
import numpy as np
import pandas as pd
import jax.numpy as jnp
from jax import jit
import adoptODE
from adoptODE import simple_simulation, train_adoptODE
import matplotlib.pyplot as plt
import copy
import jax
import optax

In [ ]:
def define_system(**kwargs_sys):
    def gen_y0():
        ini_state = jnp.ones(kwargs_sys['D'], dtype=jnp.float32)
        # ini_state = jnp.ones(kwargs_sys['D'], dtype=jnp.float32)*2
        ini_state = ini_state.at[0].set(1.1)
        return {'state': ini_state}

    def gen_params():
        return {}, {}, {}

    @jit
    def eom(y, t, params, iparams, exparams):
        return {'state': (jnp.roll(y['state'], -1) - jnp.roll(y['state'], 2)) * jnp.roll(y['state'], 1) - y['state'] + params['p']}

    @jit
    def loss(ys, params, iparams, exparams, targets):
        x = ys["state"][:kwargs_sys["len_segs"]
                        ][:, ::kwargs_sys["oberserve_every"]]
        t_x = targets["state"][:kwargs_sys["len_segs"]
                               ][:, ::kwargs_sys["oberserve_every"]]
        # jax.debug.print("mean: {x}",x=jnp.nanmean((x - t_x) ** 2))
        return jnp.nanmean((x - t_x) ** 2)

    return eom, loss, gen_params, gen_y0, {}

In [ ]:
kwargs_sys = {
    'num_iter': 100,
    'init_range': [-1, 4],
    'param_range': [4, 20],
    'N_sys': 1,
    'D': 9,
    'p': 8.17,
    "trans_steps": 1000,
    "N_time_steps": 1000,
    "dt": 0.01,
    "len_segs": 40,
    "oberserve_every": 3,
    "seed": 0,
    'lr_init': 0.1,
    'lr': 0.01
}

t_evals = jnp.arange(
    0, (kwargs_sys["N_time_steps"]+kwargs_sys["trans_steps"])*kwargs_sys["dt"], kwargs_sys["dt"])
num_segs = int(kwargs_sys['N_time_steps']/kwargs_sys['len_segs'])

kwargs_adoptODE = {'lr': kwargs_sys['lr'],
                   'epochs': 3000,
                   'lr_y0': 0,
                   'custom_scheduel_y0': optax.cosine_decay_schedule(kwargs_sys['lr_init'], 3000, alpha=1e-3, exponent=1.0)
                   # 'optimizer_y0':sgd_bounded,
                   # 'optimizer_y0_kwargs':{}
                   }

dataset = simple_simulation(define_system,
                            t_evals,
                            kwargs_sys,
                            kwargs_adoptODE,
                            params={'p': kwargs_sys['p']})

In [ ]:
# make a deep copy to keep the ground truth data
dataset_gt = copy.deepcopy(dataset)

# remove the transient phase from the ground truth data
dataset_gt.ys["state"] = dataset_gt.ys["state"][:, kwargs_sys["trans_steps"]:]
dataset_gt.y0["state"] = dataset_gt.ys["state"][:, 0, :]
dataset_gt.t_evals = jnp.arange(
    0, (kwargs_sys["N_time_steps"])*kwargs_sys["dt"], kwargs_sys["dt"])

In [ ]:
def gen_dataset(dataset_gt: adoptODE.Framework.dataset_adoptODE, kwargs_sys: dict, params: np.array, seg_number):

    dataset = copy.deepcopy(dataset_gt)
    dataset.y0["state"] = copy.deepcopy(dataset_gt.ys["state"][:, seg_number*kwargs_sys["len_segs"]])
    mask = np.zeros(dataset.ys["state"].shape, dtype=bool)
    mask[:, :, ::kwargs_sys["oberserve_every"]] = 1
    mask_y0 = mask[:, 0, :]

    dataset.t_evals = dataset.t_evals[:kwargs_sys["len_segs"]]
    dataset.ys["state"] = dataset.ys["state"]*jnp.nan
    dataset.ys["state"][mask] = dataset_gt.ys["state"][mask]
    dataset.ys["state"] = dataset.ys["state"][:, seg_number*kwargs_sys["len_segs"]:(seg_number+1)*kwargs_sys["len_segs"]]

    # dataset.y0_train["state"]=dataset.y0_train["state"]*0
    dataset.y0_train["state"][~mask_y0] = params
    dataset.y0_train["state"][mask_y0] = dataset.y0["state"][mask_y0]

    y0_lower_bound = jnp.full(dataset.y0_train["state"].shape, -jnp.inf)
    y0_lower_bound = y0_lower_bound.at[mask_y0].set(
        dataset.y0["state"][mask_y0])

    y0_upper_bound = jnp.full(dataset.y0_train["state"].shape, jnp.inf)
    y0_upper_bound = y0_upper_bound.at[mask_y0].set(
        dataset.y0["state"][mask_y0])

    dataset.kwargs_adoptODE['lower_b_y0'] = {'state': y0_lower_bound}
    dataset.kwargs_adoptODE['upper_b_y0'] = {'state': y0_upper_bound}

    return dataset

In [ ]:
dir = os.path.join("results")
os.makedirs(dir)

init = np.zeros((num_segs, kwargs_sys["D"], kwargs_sys['num_iter']))
mse_true = np.zeros((num_segs, kwargs_sys['num_iter']))
mse_measured = np.zeros((num_segs, kwargs_sys['num_iter']))
p_estimated = np.zeros((num_segs, kwargs_sys['num_iter']))
key = jax.random.key(kwargs_sys["seed"])
    
for seg_number in range(num_segs):
    true = dataset_gt.ys['state'][:, seg_number*kwargs_sys["len_segs"]:(seg_number+1)*kwargs_sys["len_segs"]]
    for i in range(kwargs_sys['num_iter']):
        print(seg_number, i)
        _, key = jax.random.split(key)
        init_params = np.array(jax.random.uniform(
        key, dataset_gt.y0_train["state"].shape, minval=kwargs_sys["init_range"][0], maxval=kwargs_sys["init_range"][1]))
        init_params = np.delete(init_params, np.s_[::kwargs_sys["oberserve_every"]], axis=-1)[0]
        dataset = gen_dataset(dataset_gt, kwargs_sys, init_params, seg_number)
        p_train = jax.random.uniform(key, minval=kwargs_sys["param_range"][0], maxval=kwargs_sys["param_range"][1])
        dataset.params_train = {'p': p_train}
        params_final, losses, errors, params_history = train_adoptODE(
            dataset, save_interval=None, print_interval=None)
        
        init[seg_number, :, i] = dataset.y0_train['state'][0]
        mse_true[seg_number, i] = np.mean((dataset.ys_sol['state'] - true)**2)
        mse_measured[seg_number, i] = np.nanmean((dataset.ys_sol['state'] - dataset.ys['state'])**2)
        p_estimated[seg_number, i] = params_final['params']['p']


np.save(os.path.join(dir, "init.npy"), init)
pd.DataFrame(mse_true).to_csv(os.path.join(dir, "mse_true.csv"), header=False, index=False)
pd.DataFrame(mse_measured).to_csv(os.path.join(dir, "mse_measured.csv"), header=False, index=False)
pd.DataFrame(p_estimated).to_csv(os.path.join(dir, "p.csv"), header=False, index=False)
pd.DataFrame(dataset_gt.ys['state'][0, :, :]).to_csv(os.path.join(dir, "true.csv"), header=False, index=False)

with open(os.path.join(dir, 'kwargs_sys.json'), 'w') as f:
    json.dump(kwargs_sys, f, indent=4)
